Delta of MDL and Surprisal across revisions (2 types)

- cumulative delta — revised_k - base (revision_depth 0), always compared with the base
- step delta — revised_k - revised_(k-1), one step at a time
- delta rounded to 5 decimal places (same precision as source csv), sign = +1 / 0 / -1 (0 only if exactly 0.00000)
- sentence level: groups are split at weak links (bleu_to_previous <= 0.06), same as build-master-sheet
- metrics (22) — mean_mdl, gpt2 mean_surprisal, 20 srilm mean_surprisal (4 corpora x n = 1..5)

Output shape (long format, one row per pair x delta_type x metric):
article, revision_group_no, group_max_depth, delta_type, depth, from_depth, metric, from_value, to_value, delta, sign

In [1]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/ to read and write (final stats/plots go to output/<LEVEL>/)
mdl_csv_path = Path(f"../../data/{LEVEL}/features/mdl-csv")
gpt2_csv_path = Path(f"../../data/{LEVEL}/features/mean-surprisal-gpt2-csv")
srilm_csv_path = Path(f"../../data/{LEVEL}/features/mean-surprisal-srilm-csv")
out_csv_path = Path(f"../../data/{LEVEL}/delta/delta-mdl-surprisal-csv") # one long csv per article
master_csv_file = Path(f"../../data/{LEVEL}/delta/delta-mdl-surprisal-long.csv") # all articles combined
out_csv_path.mkdir(parents=True, exist_ok=True)

# sentence level only: a group is split wherever bleu_to_previous <= BLEU_THRESHOLD (same rule as build-master-sheet),
# so no delta compares a sentence with an unrelated one. Paragraph pairs were already filtered in data-preprocessing.
BLEU_THRESHOLD = 0.06 if LEVEL == "sentence" else None

out_column_names = ["article", "revision_group_no", "group_max_depth", "delta_type", "depth", "from_depth",
                    "metric", "from_value", "to_value", "delta", "sign"]

In [2]:
def read_metric_rows(csv_file_name):
    #read same article from mdl, gpt2 and srilm folders and merge into one row per (group, depth) with all 22 metrics
    #returns list of dicts: {revision_group_no, revision_depth, text, metric_name: value, ...}
    with (mdl_csv_path / csv_file_name).open("r", encoding="utf-8") as f:
       mdl_rows = list(csv.DictReader(f))
    with (gpt2_csv_path / csv_file_name).open("r", encoding="utf-8") as f:
       gpt2_rows = list(csv.DictReader(f))
    with (srilm_csv_path / csv_file_name).open("r", encoding="utf-8") as f:
       srilm_rows = list(csv.DictReader(f))
    assert len(mdl_rows) == len(gpt2_rows) == len(srilm_rows), f"row count mismatch in {csv_file_name}"

    merged_rows = []
    for mdl_row, gpt2_row, srilm_row in zip(mdl_rows, gpt2_rows, srilm_rows):
        key = (mdl_row["revision_group_no"], mdl_row["revision_depth"], mdl_row["text"])
        #all 3 files must be about the same paragraph revision, otherwise the delta is meaningless
        assert key == (gpt2_row["revision_group_no"], gpt2_row["revision_depth"], gpt2_row["text"]), f"gpt2 misaligned in {csv_file_name}"
        assert key == (srilm_row["revision_group_no"], srilm_row["revision_depth"], srilm_row["text"]), f"srilm misaligned in {csv_file_name}"
        row = {"revision_group_no": int(mdl_row["revision_group_no"]),
               "revision_depth": int(mdl_row["revision_depth"]),
               "text": mdl_row["text"]}
        if BLEU_THRESHOLD is not None:
            row["bleu"] = mdl_row["bleu_to_previous"] # "" at depth 0
        row["mdl"] = float(mdl_row["mean_mdl"])
        row["surprisal_gpt2"] = float(gpt2_row["mean_surprisal"])
        for column in srilm_row:
            if column.startswith("mean_surprisal_"): # mean_surprisal_brown_1 -> surprisal_srilm_brown_1
               row["surprisal_srilm_" + column[len("mean_surprisal_"):]] = float(srilm_row[column])
        merged_rows.append(row)
    return merged_rows

In [3]:
def get_metric_names(row):
    return [key for key in row if key not in ("revision_group_no", "revision_depth", "text", "bleu")]

def group_by_revision_group(rows):
    #revision_group_no -> list of rows sorted by revision_depth (0 = base, 1, 2, ...)
    groups = {}
    for row in rows:
        groups.setdefault(row["revision_group_no"], []).append(row)
    for group_no in groups:
        groups[group_no].sort(key=lambda r: r["revision_depth"])
        depths = [r["revision_depth"] for r in groups[group_no]]
        assert depths == list(range(len(depths))), f"group {group_no} depths are not 0..k: {depths}"
    return groups

def split_weak_links(groups):
    #sentence level: cut a group wherever bleu_to_previous <= BLEU_THRESHOLD (same as build-master-sheet.ipynb)
    #rows after a weak link become a new group (new number = max + 1, ...) with depths renumbered 0, 1, ...
    next_no = max(groups, default=0) + 1
    result = {}
    for group_no in sorted(groups):
        segment, current_no = [], group_no
        for row in groups[group_no]:
            bleu = row.get("bleu", "") # "" = base row / no BLEU info
            if row["revision_depth"] > 0 and bleu != "" and float(bleu) <= BLEU_THRESHOLD and segment:
                result[current_no] = segment # close the group before the weak link
                current_no, next_no = next_no, next_no + 1
                segment = []
            segment.append({**row, "revision_depth": len(segment)})
        result[current_no] = segment
    return result


In [4]:
def get_delta_and_sign(from_value, to_value):
    #round to 5 decimal places (source csv precision) so float noise like 1e-17 does not count as a change
    delta = round(to_value - from_value, 5) + 0.0 # + 0.0 turns -0.0 into 0.0
    if delta > 0:
        sign = 1
    elif delta < 0:
        sign = -1
    else: # exactly 0.00000
        sign = 0
    return delta, sign

In [5]:
def calculate_deltas(article, rows):
    #returns list of output rows (long format) for one article
    if not rows: # sentence level: some articles have no sentence groups at all (header only csv)
        return []
    groups = group_by_revision_group(rows)
    if BLEU_THRESHOLD is not None:
        groups = split_weak_links(groups)
    metric_names = get_metric_names(rows[0])
    out_rows = []
    for group_no, group_rows in groups.items():
        group_max_depth = len(group_rows) - 1
        for depth in range(1, group_max_depth + 1): # depth 0 is the base, nothing to compare with
            #cumulative: compare with base (depth 0), step: compare with previous revision (depth - 1)
            for delta_type, from_depth in (("cumulative", 0), ("step", depth - 1)):
                from_row = group_rows[from_depth]
                to_row = group_rows[depth]
                for metric in metric_names:
                    delta, sign = get_delta_and_sign(from_row[metric], to_row[metric])
                    out_rows.append({"article": article,
                                     "revision_group_no": group_no,
                                     "group_max_depth": group_max_depth,
                                     "delta_type": delta_type,
                                     "depth": depth,
                                     "from_depth": from_depth,
                                     "metric": metric,
                                     "from_value": f"{from_row[metric]:.5f}",
                                     "to_value": f"{to_row[metric]:.5f}",
                                     "delta": f"{delta:.5f}",
                                     "sign": sign})
    return out_rows

In [6]:
for csv_file in sorted(mdl_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    article = csv_file.stem
    rows = read_metric_rows(csv_file.name)
    out_rows = calculate_deltas(article, rows)
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".csv.tmp")
    with tmp_csv_file.open("w", encoding="utf-8", newline="") as of:
       csv_writer = csv.DictWriter(of, fieldnames=out_column_names)
       csv_writer.writeheader()
       csv_writer.writerows(out_rows)
    tmp_csv_file.rename(out_csv_file)
    #break #- During demo we can uncomment and providea demo path

In [ ]:
# COMBINE ALL ARTICLES INTO ONE MASTER LONG CSV
tmp_master_csv_file = master_csv_file.with_suffix(".csv.tmp")
with tmp_master_csv_file.open("w", encoding="utf-8", newline="") as of:
   csv_writer = csv.writer(of)
   csv_writer.writerow(out_column_names)
   for csv_file in sorted(out_csv_path.glob("*.csv")):
       with csv_file.open("r", encoding="utf-8") as cf:
          csv_reader = csv.reader(cf)
          next(csv_reader) # skip header
          csv_writer.writerows(csv_reader)
tmp_master_csv_file.rename(master_csv_file)
print(f"written: {master_csv_file}")

In [8]:
# TEST DELTA CODE

# rounding - float noise must be 0, -0.0 must not appear
assert get_delta_and_sign(2.0, 2.0) == (0.0, 0)
assert get_delta_and_sign(0.1 + 0.2, 0.3) == (0.0, 0) # 0.3 - 0.30000000000000004 is noise
assert str(get_delta_and_sign(3.5, 3.5)[0]) == "0.0" # not "-0.0"
assert get_delta_and_sign(3.22727, 3.50000) == (0.27273, 1)
assert get_delta_and_sign(5.82163, 3.77553) == (-2.0461, -1)
assert get_delta_and_sign(1.00000, 1.00001) == (1e-05, 1) # smallest change at 5 dp still counts

# toy group with 3 revisions: cumulative and step must differ at depth 2, same at depth 1
toy_rows = [{"revision_group_no": 1, "revision_depth": d, "text": "", "mdl": v} for d, v in enumerate([2.0, 3.0, 2.5])]
toy_out = {(r["delta_type"], r["depth"]): (r["from_depth"], r["delta"], r["sign"]) for r in calculate_deltas("toy", toy_rows)}
assert toy_out[("cumulative", 1)] == (0, "1.00000", 1)
assert toy_out[("step", 1)] == (0, "1.00000", 1)
assert toy_out[("cumulative", 2)] == (0, "0.50000", 1)   # 2.5 - 2.0
assert toy_out[("step", 2)] == (1, "-0.50000", -1)       # 2.5 - 3.0
# sum of step deltas = cumulative delta at last depth
assert abs(float(toy_out[("step", 1)][1]) + float(toy_out[("step", 2)][1]) - float(toy_out[("cumulative", 2)][1])) < 1e-9
# weak link split (sentence level): 0.03 <= 0.06 cuts the group before depth 2, depths restart at 0
if BLEU_THRESHOLD is not None:
    toy = {1: [{"revision_group_no": 1, "revision_depth": d, "text": "", "bleu": b, "mdl": v}
               for d, (b, v) in enumerate([("", 2.0), ("0.5", 3.0), ("0.03", 9.0), ("0.4", 8.0)])],
           2: [{"revision_group_no": 2, "revision_depth": 0, "text": "", "bleu": "", "mdl": 1.0}]}
    split = split_weak_links(toy)
    assert sorted(split) == [1, 2, 3]
    assert [r["mdl"] for r in split[1]] == [2.0, 3.0] and [r["mdl"] for r in split[3]] == [9.0, 8.0]
    assert [r["revision_depth"] for r in split[3]] == [0, 1]
    assert get_metric_names(split[1][0]) == ["mdl"] # bleu is never treated as a metric

print("all delta checks passed")


all delta checks passed


In [9]:
# DEMO - HOW CUMULATIVE AND STEP DELTA WORK FOR ONE ARTICLE
# step 1: read mdl, gpt2 and srilm csv of the article and merge row by row
# step 2: group rows by revision_group_no, sorted by revision_depth (0 = base)
# step 3: for every depth k >= 1: cumulative = value_k - value_0, step = value_k - value_(k-1)
# step 4: round delta to 5 dp, sign = +1 / 0 / -1

demo_article = "Fake-Death"
demo_metrics = ["mdl", "surprisal_gpt2"] # keep demo readable, all 22 metrics are in the output csv

demo_rows = read_metric_rows(f"{demo_article}.csv")
demo_groups = group_by_revision_group(demo_rows)

for group_no, group_rows in demo_groups.items():
    if len(group_rows) < 3: # show only groups with 2+ revisions so cumulative and step look different
       continue
    print(f"\n=== Group {group_no} (max depth {len(group_rows) - 1})")
    for row in group_rows:
        print(f"  depth {row['revision_depth']}: " + ", ".join(f"{m}={row[m]:.5f}" for m in demo_metrics) + f" | {row['text'][:70]}")
    print(f"  {'METRIC':16} {'DEPTH':6} {'TYPE':11} {'FROM':5} {'FROM_VAL':>9} {'TO_VAL':>9} {'DELTA':>9} SIGN")
    for metric in demo_metrics:
        for depth in range(1, len(group_rows)):
            for delta_type, from_depth in (("cumulative", 0), ("step", depth - 1)):
                from_value = group_rows[from_depth][metric]
                to_value = group_rows[depth][metric]
                delta, sign = get_delta_and_sign(from_value, to_value)
                print(f"  {metric:16} {depth:<6} {delta_type:11} {from_depth:<5} {from_value:9.5f} {to_value:9.5f} {delta:9.5f} {sign:+d}")

#same numbers must come from the real function
demo_out = calculate_deltas(demo_article, demo_rows)
print(f"\n{demo_article}: {len(demo_rows)} revisions, {len(demo_groups)} groups -> {len(demo_out)} delta rows")


=== Group 2 (max depth 2)
  depth 0: mdl=3.22727, surprisal_gpt2=6.20215 | Make sure you hold your breath as long as you can then when their not 
  depth 1: mdl=3.50000, surprisal_gpt2=5.60301 | Make sure you hold your breath as long as you can then when their not 
  depth 2: mdl=2.82609, surprisal_gpt2=4.17762 | Make sure you hold your breath as long as you can, then when they're n
  METRIC           DEPTH  TYPE        FROM   FROM_VAL    TO_VAL     DELTA SIGN
  mdl              1      cumulative  0       3.22727   3.50000   0.27273 +1
  mdl              1      step        0       3.22727   3.50000   0.27273 +1
  mdl              2      cumulative  0       3.22727   2.82609  -0.40118 -1
  mdl              2      step        1       3.50000   2.82609  -0.67391 -1
  surprisal_gpt2   1      cumulative  0       6.20215   5.60301  -0.59914 -1
  surprisal_gpt2   1      step        0       6.20215   5.60301  -0.59914 -1
  surprisal_gpt2   2      cumulative  0       6.20215   4.17762  -2.0245

In [10]:
# SUMMARY - AT EVERY DEPTH: HOW MANY DELTA +/0/- AND NET DELTA CHANGE
import pandas as pd

delta_df = pd.read_csv(master_csv_file, dtype={"article": "category", "delta_type": "category", "metric": "category"})
print(f"rows: {len(delta_df)}")

def depth_summary(df, metric, delta_type):
    subset = df[(df["metric"] == metric) & (df["delta_type"] == delta_type)]
    by_depth = subset.groupby("depth")
    summary = by_depth["sign"].value_counts().unstack(fill_value=0).reindex(columns=[1, 0, -1], fill_value=0)
    summary.columns = ["positive", "zero", "negative"]
    summary["total"] = summary.sum(axis=1)
    summary["net_count"] = summary["positive"] - summary["negative"] # direction: + means more increases
    summary["net_delta_sum"] = by_depth["delta"].sum().round(5)      # net delta change
    summary["mean_delta"] = by_depth["delta"].mean().round(5)
    summary["median_delta"] = by_depth["delta"].median().round(5)
    return summary

for metric in ["mdl", "surprisal_gpt2"]:
    for delta_type in ["cumulative", "step"]:
        print(f"\n=== {metric} - {delta_type}")
        print(depth_summary(delta_df, metric, delta_type).head(10))

# demo for one article: depth_summary(delta_df[delta_df["article"] == "Fake-Death"], "mdl", "step")

rows: 7943584

=== mdl - cumulative
       positive   zero  negative   total  net_count  net_delta_sum  \
depth                                                                
1         54636  32775     47615  135026       7021     1063.09529   
2         14507   2222     12898   29627       1609      397.04896   
3          4684    246      4009    8939        675      175.61271   
4          1842     42      1626    3510        216       97.11671   
5           849     11       734    1594        115       47.57153   
6           450      5       336     791        114       44.00880   
7           239      0       174     413         65       27.48865   
8           140      0       104     244         36       15.99322   
9            78      0        57     135         21        7.75966   
10           54      0        41      95         13        8.19465   

       mean_delta  median_delta  
depth                            
1         0.00787       0.00000  
2         0.01340    

Per depth analysis — average delta of +ve, -ve, 0 groups and overall change

For every metric x delta_type x depth (plus an `all` row = all depths together):
- n_pos / n_zero / n_neg and their % — how many deltas went up / stayed same / went down
- mean_pos_delta / mean_neg_delta — average size of the change inside the +ve and -ve group (0 group is always 0)
- sum_pos / sum_neg / net_delta_sum — total increase, total decrease and overall (net) change
- mean_delta / median_delta — overall average change (net_delta_sum / n), mean_abs_delta — size ignoring direction

Output: output/paragraph/tables/delta-per-depth-stats.csv

In [ ]:
tables_dir = Path(f"../../output/{LEVEL}/tables")
figures_dir = Path(f"../../output/{LEVEL}/figures")
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

def sign_group_stats(delta):
    #delta = pandas series of rounded deltas for one (metric, delta_type, depth)
    pos = delta[delta > 0]
    neg = delta[delta < 0]
    n = len(delta)
    return {"n": n,
            "n_pos": len(pos), "n_zero": int((delta == 0).sum()), "n_neg": len(neg),
            "pct_pos": 100 * len(pos) / n, "pct_zero": 100 * (delta == 0).sum() / n, "pct_neg": 100 * len(neg) / n,
            "mean_pos_delta": pos.mean() if len(pos) else 0.0, # average increase among the +ve group
            "mean_neg_delta": neg.mean() if len(neg) else 0.0, # average decrease among the -ve group
            "sum_pos": pos.sum(), "sum_neg": neg.sum(),
            "net_delta_sum": delta.sum(),                     # overall change = sum_pos + sum_neg
            "mean_delta": delta.mean(),                        # overall average change
            "median_delta": delta.median(),
            "mean_abs_delta": delta.abs().mean()}

stat_rows = []
for (metric, delta_type), subset in delta_df.groupby(["metric", "delta_type"], observed=True):
    for depth, depth_subset in subset.groupby("depth"):
        stat_rows.append({"metric": metric, "delta_type": delta_type, "depth": str(depth), **sign_group_stats(depth_subset["delta"])})
    stat_rows.append({"metric": metric, "delta_type": delta_type, "depth": "all", **sign_group_stats(subset["delta"])})

depth_stats = pd.DataFrame(stat_rows).round(5)
depth_stats.to_csv(tables_dir / "delta-per-depth-stats.csv", index=False)
print(f"written: {tables_dir / 'delta-per-depth-stats.csv'} ({len(depth_stats)} rows)")

# show 3 headline metrics (srilm wikitext2_1 = best in exp 1, gpt2, mdl), depth 1..10 + all
show_columns = ["depth", "n", "pct_pos", "pct_zero", "pct_neg", "mean_pos_delta", "mean_neg_delta", "net_delta_sum", "mean_delta", "median_delta"]
pd.set_option("display.width", 250)
for metric in ["surprisal_srilm_wikitext2_1", "surprisal_gpt2", "mdl"]:
    for delta_type in ["cumulative", "step"]:
        view = depth_stats[(depth_stats.metric == metric) & (depth_stats.delta_type == delta_type)]
        view = view[view.depth.isin([str(d) for d in range(1, 11)] + ["all"])]
        print(f"\n=== {metric} - {delta_type}")
        print(view[show_columns].to_string(index=False))

Plot per depth — mean delta of the +ve group, the -ve group and all pairs (net), at every depth and overall

One figure per metric (22) x delta type (step / cumulative):
- left — mean Δ at every depth for the Δ > 0 group, the Δ < 0 group and all pairs together (net); the Δ = 0 group is 0 by definition so it is only in the table
- right — the same means over all depths pooled (pair-weighted, = the `all` row of delta-per-depth-stats.csv); no unweighted (mean of depth means) version
- bottom — the numbers behind the plot: mean Δ, count and % of pairs per group, last column = all depths

Only depths with at least 50 pairs are plotted. Reads delta-per-depth-stats.csv, so this cell can be re-run on its own.

Output: output/<LEVEL>/figures/delta-by-depth/<step|cumulative>/<metric>.png

In [ ]:
# PLOT - MEAN DELTA OF THE +VE / -VE GROUP AND NET AT EVERY DEPTH, PLUS THE OVERALL (ALL DEPTHS) MEAN
# one figure per metric x delta_type, read from delta-per-depth-stats.csv (so this cell can run on its own)
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

MIN_PAIRS = 50  # a depth is plotted only if it has at least this many pairs (fewer = mean is mostly noise)
plot_dir = figures_dir / "delta-by-depth"

UP, DOWN, NET = "#e34948", "#2a78d6", "#1f1f1f"  # red / blue checked for colour-blind separation, net = ink
INK, INK_2, GRID = "#1f1f1f", "#5f5e5a", "#e6e5e1"
TINT = {"Δ > 0": "#fbe3e2", "Δ < 0": "#deeafa", "Δ = 0": "#f0efec", "Net": "#e4e3df"}

CORPUS = {"brown": "Brown", "ptb_wsj": "PTB-WSJ", "simplewiki": "Simple Wikipedia", "wikitext2": "WikiText-2"}
DELTA_TEXT = {"step": "step Δ = v(k) − v(k−1)", "cumulative": "cumulative Δ = v(k) − v(0)"}


def metric_title(metric):
    if metric == "mdl":
        return "Mean dependency length"
    if metric == "surprisal_gpt2":
        return "Surprisal (GPT-2)"
    corpus, order = metric[len("surprisal_srilm_"):].rsplit("_", 1)
    return f"Surprisal (SRILM · {CORPUS[corpus]} · {order}-gram)"


def plot_one(stats, metric, delta_type, out_file):
    rows = stats[(stats.metric == metric) & (stats.delta_type == delta_type)]
    overall = rows[rows.depth == "all"].iloc[0]  # all depths pooled = pair-weighted overall
    per_depth = rows[rows.depth != "all"].copy()
    per_depth["depth"] = per_depth["depth"].astype(int)
    per_depth = per_depth[per_depth.n >= MIN_PAIRS].sort_values("depth")
    depths = per_depth.depth.tolist()

    with matplotlib.rc_context({"font.size": 9, "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK_2,
                                "xtick.color": INK_2, "ytick.color": INK_2, "axes.titlecolor": INK}):
        fig = plt.figure(figsize=(13, 8.2))
        grid = fig.add_gridspec(2, 2, width_ratios=[3.3, 1], height_ratios=[2.5, 1.35],
                                hspace=0.2, wspace=0.16, left=0.07, right=0.98, top=0.86, bottom=0.03)
        ax = fig.add_subplot(grid[0, 0])
        ax_all = fig.add_subplot(grid[0, 1], sharey=ax)
        ax_table = fig.add_subplot(grid[1, :])

        fig.text(0.07, 0.955, f"{LEVEL.capitalize()} · {metric_title(metric)}", fontsize=14, weight="bold", color=INK)
        fig.text(0.07, 0.918, f"{DELTA_TEXT[delta_type]}  ·  mean Δ inside each sign group  ·  "
                 f"depths with ≥ {MIN_PAIRS} pairs  ·  n = {int(overall.n):,} pairs", fontsize=10, color=INK_2)

        # left: per depth lines (the Δ = 0 group is 0 by definition, so it is only in the table)
        ax.axhline(0, color="#9d9c97", lw=0.9, zorder=1)
        ax.plot(depths, per_depth.mean_pos_delta, color=UP, lw=2, marker="o", ms=6, label="Δ > 0  (went up)", zorder=3)
        ax.plot(depths, per_depth.mean_neg_delta, color=DOWN, lw=2, marker="s", ms=6, label="Δ < 0  (went down)", zorder=3)
        ax.plot(depths, per_depth.mean_delta, color=NET, lw=2.4, marker="D", ms=6, label="Net  (all pairs)", zorder=4)
        ax.set_xticks(depths)
        ax.set_xlim(depths[0] - 0.5, depths[-1] + 0.5)
        ax.set_xlabel("Revision depth  k")
        ax.set_ylabel("Mean Δ")
        ax.set_title("By revision depth", loc="left", fontsize=10.5, weight="bold", pad=8)

        # right: overall = every pair at every depth pooled (pair-weighted)
        bars = [("Net", overall.mean_delta, NET), ("Δ > 0", overall.mean_pos_delta, UP), ("Δ < 0", overall.mean_neg_delta, DOWN)]
        ax_all.axhline(0, color="#9d9c97", lw=0.9, zorder=1)
        ax_all.bar(range(3), [b[1] for b in bars], width=0.56, color=[b[2] for b in bars], zorder=2)
        ax_all.set_xticks(range(3), [b[0] for b in bars])
        ax_all.tick_params(axis="y", labelleft=False)
        ax_all.set_xlim(-0.6, 2.6)
        ax_all.set_title("All depths (pair-weighted)", loc="left", fontsize=10.5, weight="bold", pad=8)

        # shared y range with room for the bar labels
        values = [*per_depth.mean_pos_delta, *per_depth.mean_neg_delta, *per_depth.mean_delta, *[b[1] for b in bars]]
        low, high = min(values + [0]), max(values + [0])
        pad = (high - low) * 0.14 or 0.05
        ax.set_ylim(low - pad, high + pad)
        offset = (high - low + 2 * pad) * 0.015
        for x, (_, value, _) in enumerate(bars):
            ax_all.text(x, value + (offset if value >= 0 else -offset), f"{value:+.4f}", ha="center",
                        va="bottom" if value >= 0 else "top", fontsize=9, color=INK, weight="bold")
        ax_all.text(0.5, -0.115, f"Δ = 0: {int(overall.n_zero):,} pairs ({overall.pct_zero:.1f}%)",
                    transform=ax_all.transAxes, ha="center", va="top", fontsize=8.5, color=INK_2)

        for axis in (ax, ax_all):
            axis.grid(axis="y", color=GRID, lw=0.8)
            axis.set_axisbelow(True)
            axis.spines[["top", "right"]].set_visible(False)

        fig.legend(handles=[Line2D([], [], color=c, lw=2, marker=m, ms=6, label=l) for c, m, l in
                            [(UP, "o", "Δ > 0  (went up)"), (DOWN, "s", "Δ < 0  (went down)"), (NET, "D", "Net  (all pairs)")]],
                   loc="upper right", bbox_to_anchor=(0.98, 0.975), ncol=3, frameon=False, fontsize=9.5,
                   handlelength=2.2, columnspacing=1.6)

        # bottom: the numbers behind the plot, last column = all depths
        columns = [str(d) for d in depths] + ["All"]
        table_rows = per_depth.to_dict("records") + [overall.to_dict()]
        cells = [
            [f"{r['mean_pos_delta']:+.4f}\n{int(r['n_pos']):,} ({r['pct_pos']:.0f}%)" for r in table_rows],
            [f"{r['mean_neg_delta']:+.4f}\n{int(r['n_neg']):,} ({r['pct_neg']:.0f}%)" for r in table_rows],
            [f"{int(r['n_zero']):,} ({r['pct_zero']:.0f}%)" for r in table_rows],
            [f"{r['mean_delta']:+.4f}\nn = {int(r['n']):,}" for r in table_rows],
        ]
        row_labels = ["Δ > 0", "Δ < 0", "Δ = 0", "Net"]
        ax_table.axis("off")
        table = ax_table.table(cellText=cells, rowLabels=row_labels, colLabels=columns, cellLoc="center",
                               rowLoc="center", bbox=[0.06, 0, 0.94, 1])
        table.auto_set_font_size(False)
        table.set_fontsize(7.4 if len(columns) > 10 else 8)
        last = len(columns) - 1
        for (r, c), cell in table.get_celld().items():
            cell.set_edgecolor("#dcdbd6")
            cell.set_linewidth(0.6)
            text = cell.get_text()
            text.set_color(INK)
            if r == 0:
                cell.set_facecolor("#f6f5f2")
                text.set_weight("bold")
                if c == -1:
                    continue
                text.set_text(("depth " if c == 0 else "") + text.get_text())
            elif c == -1:
                cell.set_width(0.06)
                cell.set_facecolor(TINT[row_labels[r - 1]])
                text.set_weight("bold")
            if c == last:
                cell.set_facecolor("#f6f5f2" if r == 0 else "#fafaf8")
                text.set_weight("bold")

        out_file.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_file, dpi=200, facecolor="white")
        plt.close(fig)


depth_stats_for_plot = pd.read_csv(tables_dir / "delta-per-depth-stats.csv", dtype={"depth": str})
written = 0
for metric in sorted(depth_stats_for_plot.metric.unique()):
    for delta_type in ["step", "cumulative"]:
        plot_one(depth_stats_for_plot, metric, delta_type, plot_dir / delta_type / f"{metric}.png")
        written += 1
print(f"written: {written} figures under {plot_dir}")

Plot distribution of Δ — the shape behind the means above

One row per headline metric, one column per delta type (all pairs, all depths). Solid line = 0, dashed line = mean Δ
(red if it went up on average, blue if down). Clipped to the 1st–99th percentile so outliers don't flatten the plot.

Output: output/<LEVEL>/figures/delta-distribution.png

In [ ]:
# PLOT - HISTOGRAM OF DELTA FOR THE 3 HEADLINE METRICS, STEP AND CUMULATIVE
import matplotlib.pyplot as plt

hist_metrics = {"surprisal_srilm_wikitext2_1": "Surprisal (SRILM · WikiText-2 · 1-gram)",
                "surprisal_gpt2": "Surprisal (GPT-2)",
                "mdl": "Mean dependency length"}
hist_types = ["step", "cumulative"]
UP, DOWN = "#e34948", "#2a78d6" # same red / blue as the delta-by-depth figures

fig, axes = plt.subplots(len(hist_metrics), len(hist_types), figsize=(12, 3.2 * len(hist_metrics)))
for row, (metric, label) in enumerate(hist_metrics.items()):
    for col, delta_type in enumerate(hist_types):
        d = delta_df.loc[(delta_df["metric"] == metric) & (delta_df["delta_type"] == delta_type), "delta"]
        low, high = d.quantile([0.01, 0.99])
        ax = axes[row, col]
        ax.hist(d.clip(low, high), bins=80, color="#8a8984")
        ax.axvline(0, color="#1f1f1f", lw=1)
        ax.axvline(d.mean(), color=UP if d.mean() > 0 else DOWN, ls="--", lw=1.8, label=f"mean Δ = {d.mean():+.4f}")
        ax.set_title(f"{label} · {delta_type} Δ", loc="left", fontsize=10, weight="bold")
        ax.set_xlabel("Δ")
        ax.legend(loc="upper right", frameon=False, fontsize=8.5)
        ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{LEVEL.capitalize()} · distribution of Δ (n = {len(d):,} pairs per panel, clipped to 1st–99th percentile)",
             x=0.01, ha="left", fontsize=12, weight="bold")
fig.tight_layout()
fig.savefig(figures_dir / "delta-distribution.png", dpi=200, facecolor="white")
plt.close(fig)
print(f"written: {figures_dir / 'delta-distribution.png'}")